# Demonstration of the visit sequence archive

## Preparation of the environment

### Imports

In [ ]:
import pickle
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import datetime, timedelta, timezone

from astropy.time import Time
import numpy as np
import pandas as pd
import sqlite3
from lsst.resources import ResourcePath

import rubin_scheduler
from rubin_sim import maf
from rubin_sim.sim_archive import vseqarchive
from rubin_sim.sim_archive.tempdb import LocalOnlyPostgresql
from rubin_scheduler.scheduler import sim_runner
from rubin_scheduler.scheduler.model_observatory import ModelObservatory
from rubin_scheduler.scheduler.example import example_scheduler
from rubin_scheduler.scheduler.utils import SchemaConverter

### Create a temporary sandbox archive

To avoid either requiring access to the production archive or metadata database, or alteration of the production archive or alteration metadata database in the course of the demonstation, create a local archive and metadata database to work with.

This will be deleted at the end of the notebook.

Create a temporary directory to be a common root for the archive and metadata database. This will be deleted later, so everything well be cleaned up.

In [ ]:
demo_dir = TemporaryDirectory()

Create a directory for the data archive, the location where we will store the actual files, including the table of visits.
Access will be managed through `lsst.resources.ResourcePath`, so the `ResourcePath` to the root of the archive is what we will ultiamtely end up with.

In [ ]:
archive_url = "file://" + demo_dir.name + "/archive/"
archive = ResourcePath(archive_url)

Make a postgresql database in another subdirectory of our demonstration base, and take a look at the connection parameters.

In [ ]:
md_database = LocalOnlyPostgresql(base_dir=demo_dir.name)
metadata_db_kwargs = md_database.psycopg2_dsn()
metadata_db_kwargs


Create an instance of `vseqarchive.VisitSequenceArchiveMetadata` to be our interface to it, and use this interface to create the schema in it what we will use to hold the metadata.

In [ ]:
md_db_schema = 'testvseqmd'
archive_metadata =  vseqarchive.VisitSequenceArchiveMetadata(
    metadata_db_kwargs=metadata_db_kwargs,
    metadata_db_schema=md_db_schema,
)
archive_metadata.create_schema_in_database()

Delete the instance of `VisitSequencArchiveMetadata` we just used to create the schema (`archive_metadata`).
We *could* just reuse this later, but I want to start the next section by creating a new instance, so lets get clean up this old one.

In [ ]:
del(archive_metadata)

## Create the interface we will use to talk to the metadata database

If we didn't just delete it, we could have continued using the previous one, but I want a clean beginning here.
Lets looks at the connection parameters we need to use:

In [ ]:
metadata_db_kwargs

We are using a local-only demonstration database here, so the `'host'` is a directory name, but in production this will be the host name for the production database.

In [ ]:
archive_metadata =  vseqarchive.VisitSequenceArchiveMetadata(
    metadata_db_kwargs=metadata_db_kwargs,
    metadata_db_schema=md_db_schema,
)

At this point we can directly query the database, but it isn't very interesting yet:

In [ ]:
archive_metadata.query("SELECT * FROM simulations")

But, we can already do things like check out what tables are in our schema:

In [ ]:
archive_metadata.query("SELECT table_name FROM information_schema.tables WHERE table_schema='testvseqmd'")

## Adding an entry for visits queried from consdb

I have a data file, `/Users/neilsen/Data/consdb/consdb_20250922133041.db`, that is the result of querying the consdb using `sv_survey.simulate_sv.fetch_previous_sv_visits`. Let's add an entry for that to the metadata database.

In [ ]:
consdb_visits_db = "/Users/neilsen/Data/consdb/consdb_20250922133041.db"
with sqlite3.connect(consdb_visits_db) as conn:
    consdb_visits = pd.DataFrame(maf.get_sim_data(conn))
consdb_visits.head()

Now, record the visit metadata in the metadata database.
This bare-bones version just saves a label and a hash of the table of visits, and nothing else.
We could, if we had it, also save the string that was the query sent, and the time the query was sent.

In [ ]:
sample_consdb_uuid = archive_metadata.record_completed_metadata(
    visits=consdb_visits,
    label="Sample query from consdb #1"
)
sample_consdb_uuid

It returns a unique identifier that the database can use to track this set of visits.
We can use this, for example, to take a look at the metadata we just added:

In [ ]:
archive_metadata.get_visitseq_metadata(sample_consdb_uuid, 'completed')

We can take a look at thewhole table that holds metadata on sequences of completed visits:

In [ ]:
archive_metadata.pd_read_sql(
    f"SELECT * FROM testvseqmd.completed",
)

## Running a simulation and adding it to the archive

Configure the scheduler and model observatory so the survey starts at the first visit in the consdb query.

In [ ]:
first_consdb_dayobs = np.min(consdb_visits.day_obs)
first_consdb_datetime = (
    datetime.strptime(str(first_consdb_dayobs), "%Y%m%d").replace(tzinfo=timezone.utc) + timedelta(hours=12)
)
first_consdb_time = Time(first_consdb_datetime)
survey_start_mjd = first_consdb_time.mjd

Begin by creating instances of the `ModelObservatory` and the scheduler we want.
For this notebook, we'll use the example scheduler.

In [ ]:
model_observatory = ModelObservatory(mjd_start=survey_start_mjd, downtimes='ideal')
scheduler = example_scheduler(mjd_start=survey_start_mjd)

Convert the visits from the consdb into a form that can be added to the scheduler, and load them in:

In [ ]:
consdb_obs = SchemaConverter().opsimdf2obs(consdb_visits)
scheduler.add_observations_array(consdb_obs)

Let's start our simulation the night after the last visit in our consdb query results. 

In [ ]:
last_consdb_dayobs = np.max(consdb_visits.day_obs)
last_consdb_datetime = (
    datetime.strptime(str(last_consdb_dayobs), "%Y%m%d").replace(tzinfo=timezone.utc) + timedelta(hours=12)
)
last_consdb_time = Time(last_consdb_datetime)
sim_start_mjd = last_consdb_time.mjd + 1
print(f"Last visit in consdb started at {np.max(consdb_visits.obs_start)} on {last_consdb_time}, MJD {last_consdb_time.mjd}")
print(f"So the simulation should start on MJD {sim_start_mjd}, or {Time(sim_start_mjd, format='mjd').iso}")

Construct a file name in which to save the result of our simulation:

In [ ]:
sim_opsim_fname = str(Path(demo_dir.name) / 'sim1_opsim.db')
sim_opsim_fname

Build a keyword args dictionary to send to `sim_runner`, and call it:

In [ ]:
sim_runner_kwargs = {
    'sim_start_mjd': sim_start_mjd,
    'sim_duration': 3,
    'filename': sim_opsim_fname,
    'verbose': True,
}
final_model_observatory, final_scheduler, simulated_obs = sim_runner(
    model_observatory,
    scheduler,
    **sim_runner_kwargs
)

Now record the simulation in the metadata database.
Note that this does *not* actually save the visits themselves in the archive.

In [ ]:
# Get the visits into the right data structure (a visits pandas.DataFrame)
simulated_visits = SchemaConverter().obs2opsim(simulated_obs)

# Actually add the metadata:
simulation_uuid = archive_metadata.record_simulation_metadata(
    visits=simulated_visits,
    label="Sample simulated visits 1",
    first_day_obs=Time(sim_start_mjd, format='mjd').datetime.date().isoformat(),
    last_day_obs=Time(sim_start_mjd + sim_runner_kwargs['sim_duration'], format='mjd').datetime.date().isoformat(),
    sim_runner_kwargs=sim_runner_kwargs,
    parent_visitseq_uuid=sample_consdb_uuid,
    parent_last_day_obs=last_consdb_datetime.date().isoformat()
)
simulation_uuid

In [ ]:
archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations')

We can also update our metadata after the fact.
For example, in the above, we didn't set the `scheduler_version`.
Do it now:

In [ ]:
archive_metadata.update_visitseq_metadata(simulation_uuid, 'scheduler_version', rubin_scheduler.__version__)
archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations')

## Adding extended metadata

The metadata database also stores some metadata outside the primary `simulations` table. One way of getting this extra metadata in single `pandas.Sequence` by querying the `simulations_extra` view:

In [ ]:
archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations_extra')

Note that the `tags`, `comments`, and `files` fields are empty, because we have not added any to the metadata database. Lets add some tags and comments:

In [ ]:
archive_metadata.tag(simulation_uuid, 'example', 'other_tag')
archive_metadata.comment(simulation_uuid, "This is a first comment.")
archive_metadata.comment(simulation_uuid, "This is a second comment.")

Now ask for the extra metadata again, this time putting it into the local `sim_extra_metadata` variable.

In [ ]:
sim_extra_metadata = archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations_extra')

Now we can see our tags:

In [ ]:
sim_extra_metadata['tags']

and comments:

In [ ]:
sim_extra_metadata['comments']

We could also just get our commends in a `pandas.DataFrame` directly:

In [ ]:
archive_metadata.get_comments(simulation_uuid)

## Save the visits themselves into the archive

The above example only records the metadata for the simulation and consdb output in the metadata database, it does not save the visits themselves into the archive.

The visits and other files can be added to an archive (based at a `lsst.resources.ResourcePath`) using `vseqarchive.add_file`.
Recall that our simulation above saved its output into a file named by the `sim_opsim_fname` above, and passed as the `filename` keyword argument to `sim_runner`. Lets add that file to the archive:

In [ ]:
visits_resource_path = vseqarchive.add_file(
    vsarch=archive_metadata,
    uuid=simulation_uuid,
    origin=sim_opsim_fname,
    file_type='visits',
    archive_base=archive
)
visits_resource_path

(Recall that we set the `archive` variable at the top of this notebook to `ResourcePath` pointing to a temporary directory that served as a sandbox for this notebook. In production it would point to the S3 bucket that holds the archived data.)

Technically, the `filetype` can be any string, so the metadata database can support new types of files without code modifications. However, possible `filetypes` should be standardized by convertion so that they are useful in practice.

The `"visits"` `filetype` is special, designating the table of visits themselves.

Now, if we look at the `simulations` entry in the metadata database, the `visitseq_url` row is set to the URL where the visits themselves can be downloaded:

In [ ]:
archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations')

To get the file of visits, use `lsst.resources.ResourcePath` directly.
For example, if we want to copy it from the archive into `my_local_visits.h5` it our local sandbox (so it gets cleaned up), we could do this:

In [ ]:
my_local_visits_path = str(Path(demo_dir.name) / 'sim1_opsim.db')
destination_visits_rp = ResourcePath(my_local_visits_path)
origin_visits_rp = ResourcePath(archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations')['visitseq_url'])
destination_visits_rp.transfer_from(origin_visits_rp, 'copy')

In this example, the `ResourcePath` URL begins with a `"file"` URL scheme and there is no hostname, because the sandbox archive is on local disk. The production archive with have an `S3` scheme and a hostname.

No we can load our visits:

In [ ]:
retrieved_visits = pd.read_hdf(my_local_visits_path, 'observations')
retrieved_visits.head()

## Saving other files

Saving the visits (`file_type = "visits"`) is special, and has its own dedicated row in the tables of visit sequences.
The metadata database and archive support arbitrary other types of files.

For example, we can save a pickle of the schedule we used.

Begin by saving the pickle of the scheduler to a local file:

In [ ]:
my_local_scheduler_pickle_path = str(Path(demo_dir.name) / 'scheduler.p')
with open(my_local_scheduler_pickle_path, 'wb') as scheduler_pickle_io:
    pickle.dump(scheduler, scheduler_pickle_io)

In [ ]:
!md5sum $my_local_scheduler_pickle_path

Now we can add it to the archive, much like we did with the visits:

In [ ]:
scheduler_resource_path = vseqarchive.add_file(
    vsarch=archive_metadata,
    uuid=simulation_uuid,
    origin=my_local_scheduler_pickle_path,
    file_type='scheduler',
    archive_base=archive
)
scheduler_resource_path

To find the URL later, we can either query the `files` table in the metadata database directly:

In [ ]:
archive_metadata.query(
    f"SELECT file_url FROM files WHERE file_type='scheduler' AND visitseq_uuid='{simulation_uuid}'"
)

or get it from the `simulations_extra` view:

In [ ]:
sim_extra_metadata = archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations_extra')
sim_extra_metadata['files']

## Stop our temporary postgresql database and clean up our temporary directory

In [ ]:
assert False

In [ ]:
md_database.stop()
demo_dir.cleanup()